In [12]:
#MESSY STUDENT DATA
from pyspark.sql import SparkSession, functions as function

# Create SparkSession
spark = SparkSession.builder.appName("StudentDataPreprocessing").getOrCreate()
csv_path = r"sample_data/student_data_part1.csv"

In [13]:
# Read as DataFrame with explicit DDL schema (schemas + DataFrameReader)
schema_ddl = "Name STRING, Gender STRING, Grade STRING, Math STRING, Science STRING, English STRING, Total STRING"
df = spark.read.csv(csv_path, header=True, schema=schema_ddl)

In [14]:
# Display the DataFrame with messy data and inconsistent formatting
print("1K messy data records of Student Marks")
df.show(truncate=False)

1K messy data records of Student Marks
+------------+------+--------+--------+--------+--------+-----+
|Name        |Gender|Grade   |Math    |Science |English |Total|
+------------+------+--------+--------+--------+--------+-----+
|'Navya'     |male  |11      |47.0    |63.0    |74.0    |184  |
|ROHAN       |F     |Grade 3 |16.0    |77      |8       |101  |
|'Aditi'     |0     |Grade 11|28 marks|43 marks|46.0    |117  |
|Myra        |Male  |7       |74      |12.0    |72      |158  |
|ISHAAN      |male  |03      |92 marks|27      |25.0    |144  |
|"""Aarav""" |  M   |Grade 9 |22 marks|87      |56      |165  |
|Aditi       |m     |Grade 1 |26 marks|22      |20      |68   |
|Saanvi      |Female|Grade 3 |57      |56.0    |91      |204  |
|"""Anika""" |1     |10      |45      |5       |48 marks|98   |
|Krishna     |M     |Grade 12|12 marks|64 marks|75.0    |151  |
|MYRA        |  F   |05      |99.0    |95.0    |75.0    |269  |
|"""Advika"""|F     |5       |91 marks|70 marks|21.0    |182  |
|

In [15]:
# HANDLING MISSING DATA AND DUPLICATES
# Simple rule: drop rows where key fields are null
df = df.dropna(subset=["Name", "Gender", "Grade"])

In [16]:
# Drop duplicate records based on logical student identity
df = df.dropDuplicates(["Name", "Gender", "Grade"])

In [17]:
# TRANSFORM – cleaning and structuring (TRANSFORM step, withColumn, expressions)
# 1) Normalize name (strip quotes/spaces, title case)
df = df.withColumn(
    "name",
    function.initcap(
        function.trim(
            function.regexp_replace(
                function.regexp_replace(function.col("Name"), "'", ""),
                '"',
                ""
            )
        )
    )
)

In [18]:
# 2) Normalize gender to Male/Female/Unknown (expressions, withColumn)
gender_clean = function.lower(function.trim(function.col("Gender")))
df = df.withColumn(
    "gender_std",
    function.when(gender_clean.isin("m", "male", "1"), function.lit("Male"))
     .when(gender_clean.isin("f", "female", "0"), function.lit("Female"))
     .otherwise(function.lit("Unknown"))
)

In [19]:
# 3) Normalize grade to integer (expressions on columns)
df = df.withColumn(
    "grade_int",
    function.regexp_extract(function.col("Grade"), r"(\d+)", 1).cast("int")
)

In [20]:
# 4) Clean subject scores: remove ' marks', cast to double
for old_col, new_col in [("Math", "math_clean"), ("Science", "science_clean"), ("English", "english_clean")]:
    df = df.withColumn(
        new_col,
        function.regexp_extract(function.col(old_col), r"(\d+\.?\d*)", 1).cast("double")
    )

In [21]:
# 5) Recompute total from cleaned subjects (withColumn, expressions)
df = df.withColumn(
    "total_clean",
    function.col("math_clean") + function.col("science_clean") + function.col("english_clean")
)

In [22]:
# Simple mean imputation example
means = df.agg(
    function.mean("math_clean").alias("m_mean"),
    function.mean("science_clean").alias("s_mean"),
    function.mean("english_clean").alias("e_mean")
).collect()[0]
df = df.fillna(
    {
        "math_clean": float(means["m_mean"]),
        "science_clean": float(means["s_mean"]),
        "english_clean": float(means["e_mean"])
    }
).withColumn(
    "total_clean",
    function.col("math_clean") + function.col("science_clean") + function.col("english_clean")
)

In [23]:
# 6) Drop original raw columns and keep only cleaned ones
df_cleaned = (
    df
    .select(
        "name",
        "gender_std",
        "grade_int",
        "math_clean",
        "science_clean",
        "english_clean",
        "total_clean"
    )
)

In [24]:
# Display schema and a sample (printSchema, show)
df_cleaned.printSchema()
df_cleaned.show(20, truncate=False)

root
 |-- name: string (nullable = true)
 |-- gender_std: string (nullable = false)
 |-- grade_int: integer (nullable = true)
 |-- math_clean: double (nullable = false)
 |-- science_clean: double (nullable = false)
 |-- english_clean: double (nullable = false)
 |-- total_clean: double (nullable = false)

+-----+----------+---------+----------+-------------+-------------+-----------+
|name |gender_std|grade_int|math_clean|science_clean|english_clean|total_clean|
+-----+----------+---------+----------+-------------+-------------+-----------+
|Aarav|Male      |11       |56.0      |26.0         |30.0         |112.0      |
|Aarav|Male      |9        |22.0      |87.0         |56.0         |165.0      |
|Aarav|Female    |10       |62.0      |41.0         |70.0         |173.0      |
|Aarav|Female    |6        |50.0      |72.0         |39.0         |161.0      |
|Aarav|Female    |4        |31.0      |7.0          |1.0          |39.0       |
|Aarav|Female    |8        |78.0      |71.0         |6

In [25]:
# INSIGHTS
# Insight 1: average total by grade
avg_by_grade = (
    df_cleaned.groupBy("grade_int")
      .agg(function.round(function.avg("total_clean"), 2).alias("avg_total"))
      .orderBy(function.col("avg_total").desc())
)
avg_by_grade.show(10, truncate=False)

+---------+---------+
|grade_int|avg_total|
+---------+---------+
|12       |162.64   |
|5        |158.44   |
|1        |155.4    |
|7        |153.01   |
|2        |151.61   |
|10       |151.45   |
|3        |149.3    |
|9        |148.49   |
|4        |147.33   |
|6        |147.07   |
+---------+---------+
only showing top 10 rows


In [26]:
# Insight 2: average subject scores by standardized gender
avg_by_gender = (
    df_cleaned.groupBy("gender_std")
      .agg(
          function.round(function.avg("math_clean"), 2).alias("avg_math"),
          function.round(function.avg("science_clean"), 2).alias("avg_science"),
          function.round(function.avg("english_clean"), 2).alias("avg_english"),
          function.round(function.avg("total_clean"), 2).alias("avg_total")
      )
)
avg_by_gender.show(truncate=False)

+----------+--------+-----------+-----------+---------+
|gender_std|avg_math|avg_science|avg_english|avg_total|
+----------+--------+-----------+-----------+---------+
|Female    |51.05   |49.48      |50.45      |150.98   |
|Male      |50.44   |50.28      |49.75      |150.48   |
+----------+--------+-----------+-----------+---------+



In [27]:
# Insight 3: top 10 students by total_clean
top_students = (
    df_cleaned.select("name", "gender_std", "grade_int", "math_clean", "science_clean", "english_clean", "total_clean")
      .orderBy(function.col("total_clean").desc())
      .limit(10)
)
top_students.show(truncate=False)

+-------+----------+---------+----------+-------------+-------------+-----------+
|name   |gender_std|grade_int|math_clean|science_clean|english_clean|total_clean|
+-------+----------+---------+----------+-------------+-------------+-----------+
|Navya  |Male      |6        |94.0      |96.0         |94.0         |284.0      |
|Vihaan |Female    |10       |99.0      |99.0         |84.0         |282.0      |
|Krishna|Female    |10       |98.0      |84.0         |97.0         |279.0      |
|Aditi  |Female    |1        |92.0      |90.0         |93.0         |275.0      |
|Shaurya|Female    |4        |100.0     |93.0         |81.0         |274.0      |
|Myra   |Female    |5        |99.0      |95.0         |75.0         |269.0      |
|Vihaan |Female    |12       |95.0      |76.0         |96.0         |267.0      |
|Ishaan |Male      |7        |90.0      |97.0         |77.0         |264.0      |
|Zara   |Male      |12       |83.0      |98.0         |83.0         |264.0      |
|Aryan  |Male   

In [28]:
# Write the DataFrame to Parquet file
output_path = r"sample_data/student_data_clean.parquet"
(
    df_cleaned.select(
        "name",
        "gender_std",
        "grade_int",
        "math_clean",
        "science_clean",
        "english_clean",
        "total_clean"
    )
    .write
    .mode("overwrite")
    .parquet(output_path)
)

In [29]:
spark.stop()